# Sesi 07 Generative Models: pipeline lengkap di Google Colab

Jalankan dari atas ke bawah dengan runtime **GPU (T4)**: *Runtime → Change runtime type → T4 GPU*.

Urutan: data → MLP classifier → AE, Sparse AE, VAE → GAN, DCGAN → DDPM → eksperimen MLP (Bagian 2) → evaluasi (Bagian IV) → figur (Bagian V).
Checkpoint dan hasil disimpan ke Google Drive supaya tidak hilang kalau runtime terputus. Kalau terputus, cukup jalankan ulang sel setup lalu lanjutkan dari sel training yang belum selesai.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DL07_WORKDIR'] = '/content/drive/MyDrive/dl07_sesi07'   # checkpoints + results
os.environ['DL07_DATA'] = '/content/data'                          # cache dataset (lokal, lebih cepat)
os.makedirs(os.environ['DL07_WORKDIR'], exist_ok=True)

In [ ]:
# Ambil kode dari repositori GitHub kamu (ganti URL).
# Alternatif: upload zip repo ke /content lalu: !unzip -q dl-sesi07-generative.zip -d /content
REPO_URL = 'https://github.com/USERNAME/dl-sesi07-generative.git'
if not os.path.exists('/content/dl-sesi07-generative'):
    !git clone -q {REPO_URL} /content/dl-sesi07-generative
%cd /content/dl-sesi07-generative
!pip install -q torchmetrics torch-fidelity

In [ ]:
!python -m src.data

## 2. Training

MLP classifier wajib pertama: modelnya dipakai untuk mengukur mode coverage GAN dan untuk eksperimen Bagian 2.

In [ ]:
!python -m src.train --model mlp

In [ ]:
!python -m src.train --model ae
!python -m src.train --model sae
!python -m src.train --model vae

In [ ]:
!python -m src.train --model gan

In [ ]:
!python -m src.train --model dcgan

In [ ]:
!python -m src.train --model ddpm

## 3. Eksperimen batasan MLP Classifier (Bagian 2, Tahap 2-5)

In [ ]:
!python -m src.mlp_experiment

## 4. Evaluasi kuantitatif (Bagian IV)

Termasuk FID untuk VAE/GAN/DCGAN/DDPM dan latency GPU + CPU. Bagian paling lama adalah sampling DDPM (5.000 sampel x 1000 langkah).

In [ ]:
!python -m src.evaluate

## 5. Figur 1-5 (Bagian V)

In [ ]:
!python -m src.figures

In [ ]:
from IPython.display import Markdown, Image, display
R = os.path.join(os.environ['DL07_WORKDIR'], 'results')
display(Markdown(open(f'{R}/metrics_table.md').read()))
for f in sorted(os.listdir(f'{R}/figures')):
    if f.endswith('.png'):
        print(f); display(Image(f'{R}/figures/{f}', width=900))

## 6. Unduh hasil

Salin folder `results/` ini ke repositori lokal kamu dan commit. Kirim juga `metrics.json`, `mlp_experiment.json`, `interpolation_metrics.json`, dan folder `figures/` untuk penulisan laporan.

In [ ]:
!cd {os.environ['DL07_WORKDIR']} && zip -qr /content/results_sesi07.zip results
from google.colab import files
files.download('/content/results_sesi07.zip')